# Surface Code Decoding Pipeline

This notebook demonstrates the modular QEC pipeline:
- Surface code circuit generation with configurable noise
- Pluggable decoder interface
- Logical error rate estimation with confidence intervals
- Threshold plot comparison across decoders

In [1]:
import stim
import pymatching
print(stim.__version__)
print(pymatching.__version__)

1.15.0
2.3.1


In [1]:
import sys
sys.path.insert(0, '.')

from qec_surface.circuits import build_surface_code, NoiseModel
from qec_surface.decoders import MWPMDecoder
from qec_surface.benchmark import estimate_logical_error_rate

noise = NoiseModel.uniform(0.01)
sc = build_surface_code(distance=3, rounds=3, noise=noise)
print(sc)

SurfaceCodeCircuit(type=surface_code:rotated_memory_x, d=3, rounds=3, noise=[depol=0.0100, meas=0.0100, reset=0.0100])


In [7]:
import sys
sys.path.insert(0, '..')  # adjust if needed

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from qec_surface.circuits import build_surface_code, NoiseModel
from qec_surface.decoders import MWPMDecoder, UnionFindDecoder, BeliefPropagationDecoder, BPOSDDecoder
from qec_surface.benchmark import sweep_noise_levels, compare_decoders

ImportError: cannot import name 'build_surface_code' from 'qec_surface.circuits' (unknown location)

## 1. Inspect a single circuit

Before running experiments, it's useful to inspect what stim actually generates.
The rotated surface code has d² data qubits and d²-1 ancilla qubits.

In [ ]:
sc = build_surface_code(
    distance=3,
    rounds=3,
    noise=NoiseModel.uniform(0.01)
)
print(sc)
print(f"Data qubits:    {sc.n_data_qubits}")
print(f"Ancilla qubits: {sc.n_ancilla_qubits}")
print(f"DEM errors:     {sc.detector_error_model().num_errors}")

# Visualize the circuit timeline (renders in Jupyter)
sc.circuit.diagram('timeline-svg')

## 2. MWPM threshold sweep

The threshold is the noise level where curves for different distances cross.
For depolarizing circuit-level noise on the rotated surface code,
theory and simulation give p_th ≈ 1% (Fowler et al. 2012).

In [ ]:
distances = [3, 5, 7, 9]
noise_levels = [0.001, 0.003, 0.005, 0.007, 0.01, 0.015, 0.02, 0.03, 0.05]
N_SAMPLES = 50_000

df_mwpm = sweep_noise_levels(
    distances=distances,
    noise_levels=noise_levels,
    decoder_cls=MWPMDecoder,
    n_samples=N_SAMPLES,
)

In [ ]:
def plot_threshold(df, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    colors = cm.viridis(np.linspace(0.2, 0.85, len(df['distance'].unique())))

    for color, d in zip(colors, sorted(df['distance'].unique())):
        sub = df[df['distance'] == d].sort_values('noise_level')
        ax.errorbar(
            sub['noise_level'],
            sub['logical_error_rate'],
            yerr=sub['error_bar'],
            marker='o', capsize=3, label=f'd = {d}', color=color
        )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Physical error rate p', fontsize=12)
    ax.set_ylabel('Logical error rate', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend()
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_threshold(df_mwpm, 'MWPM — Rotated Surface Code (uniform noise)')

## 3. Decoder comparison

Compare MWPM, Union-Find, and BP on the same noise sweep.

Expected behavior:
- **MWPM**: highest threshold (~1%)
- **Union-Find**: slightly lower threshold, but faster per shot
- **BP**: noticeably lower threshold due to short cycles in the surface code Tanner graph
- **BP+OSD**: close to MWPM, but much slower

In [ ]:
# Use a single distance to make the comparison fast
COMPARISON_DISTANCE = 5

df_comparison = compare_decoders(
    distances=[COMPARISON_DISTANCE],
    noise_levels=noise_levels,
    decoder_classes=[MWPMDecoder, UnionFindDecoder, BeliefPropagationDecoder, BPOSDDecoder],
    n_samples=N_SAMPLES,
)

In [ ]:
def plot_decoder_comparison(df, distance, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    sub = df[df['distance'] == distance]

    for decoder_name in sub['decoder_name'].unique():
        d_sub = sub[sub['decoder_name'] == decoder_name].sort_values('noise_level')
        ax.errorbar(
            d_sub['noise_level'],
            d_sub['logical_error_rate'],
            yerr=d_sub['error_bar'],
            marker='o', capsize=3, label=decoder_name
        )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Physical error rate p', fontsize=12)
    ax.set_ylabel('Logical error rate', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend()
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_decoder_comparison(
    df_comparison,
    distance=COMPARISON_DISTANCE,
    title=f'Decoder comparison — d={COMPARISON_DISTANCE}, uniform noise'
)

## 4. Noise model comparison (same decoder)

How does the threshold change when measurement errors dominate vs gate errors?

This is physically relevant: superconducting qubits often have measurement
fidelity as the bottleneck, while trapped ions typically have worse 2-qubit gates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
noise_models = {
    'Uniform (p_gate = p_meas)': NoiseModel.uniform,
    'Gate-only (perfect measurement)': NoiseModel.gate_only,
}

for ax, (label, factory) in zip(axes, noise_models.items()):
    colors = cm.viridis(np.linspace(0.2, 0.85, len(distances)))
    df = sweep_noise_levels(
        distances=distances,
        noise_levels=noise_levels,
        decoder_cls=MWPMDecoder,
        n_samples=N_SAMPLES,
        noise_factory=factory,
    )
    for color, d in zip(colors, sorted(df['distance'].unique())):
        sub = df[df['distance'] == d].sort_values('noise_level')
        ax.errorbar(
            sub['noise_level'], sub['logical_error_rate'],
            yerr=sub['error_bar'], marker='o', capsize=3,
            label=f'd={d}', color=color
        )
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Physical error rate p')
    ax.set_ylabel('Logical error rate')
    ax.set_title(label)
    ax.legend(); ax.grid(True, which='both', alpha=0.3)

plt.suptitle('MWPM: noise model comparison', fontsize=13)
plt.tight_layout()
plt.show()